# Workitem Signing with APS CLI 

Demonstrates the signing flow using **only CLI commands** — no SDK imports needed. Mirrors `05_workitem_signing.ipynb` but driven from the terminal, making it easy to script in CI pipelines or shell scripts.

APS public activities require a **signed workitem**: generate an RSA key pair, upload the public key to `forgeapps/me`, then sign the activity ID with the private key before each submission. APS verifies the signature server-side.

**Flow:**
1. Validate setup and `.env`
2. Define CLI helper
3. Generate key pair
4. Set nickname + upload public key
5. Verify profile
6. Sign activity ID
7. *(Optional)* Clean up local keys

**Prerequisite:** `uv run aps-automation` must be reachable from the repository root.


## 1) Preconditions

Add these to a `.env` file at the repo root:

```
CLIENT_ID=your_aps_client_id
CLIENT_SECRET=your_aps_client_secret
APS_NICKNAME=YourNickname123   # letters/numbers/underscores, max 20 chars, globally unique
```

> The CLI calls `load_dotenv(override=False)` internally. Notebooks cannot handle interactive prompts, so the `.env` must be present before running.


In [ ]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=False)

ROOT = Path.cwd().resolve()
KEY_DIR = ROOT / "keys"
PRIVATE_KEY = KEY_DIR / "mykey_cli.json"
PUBLIC_KEY = KEY_DIR / "mypublickey_cli.json"

CLIENT_ID = os.getenv("CLIENT_ID", "").strip()
CLIENT_SECRET = os.getenv("CLIENT_SECRET", "").strip()
APS_NICKNAME = os.getenv("APS_NICKNAME", "").strip()

print("Repo root:", ROOT)
print("CLIENT_ID configured:", bool(CLIENT_ID))
print("CLIENT_SECRET configured:", bool(CLIENT_SECRET))
print("APS_NICKNAME:", APS_NICKNAME or "<missing>")

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Missing CLIENT_ID/CLIENT_SECRET in .env or environment")


## 2) Helper to run CLI commands

`run_cli(args)` wraps `subprocess.run`. It prepends `uv run aps-automation`, prints the command, streams stdout/stderr, raises on non-zero exit, and returns the stdout string for downstream cells to capture (e.g. a signature).


In [ ]:
def run_cli(args: list[str]) -> str:
    cmd = ["uv", "run", "aps-automation", *args]
    print("$", " ".join(cmd))
    completed = subprocess.run(cmd, check=True, text=True, capture_output=True)
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed.stdout.strip()


## 3) Generate private key and export public key

Creates two files in `keys/`:
- `mykey_cli.json` — private key (keep secret, never commit)
- `mypublickey_cli.json` — public key to upload to APS


In [ ]:
KEY_DIR.mkdir(parents=True, exist_ok=True)

run_cli(["signing", "generate", "--keyfile", str(PRIVATE_KEY)])
run_cli(["signing", "export", "--keyfile", str(PRIVATE_KEY), "--pubkeyfile", str(PUBLIC_KEY)])

print("Private key:", PRIVATE_KEY)
print("Public key:", PUBLIC_KEY)


## 4) Set nickname and upload public key

`--nickname` sets the APS app nickname first, then uploads the public key. APS will use this key to verify signatures on every signed workitem submission.


In [ ]:
profile_json = run_cli([
    "public-key",
    "upload",
    "--pubkeyfile",
    str(PUBLIC_KEY),
    "--nickname",
    APS_NICKNAME,
])

print("Upload/profile response captured.")


## 5) Verify profile (`forgeapps/me`)

Reads the current APS app profile and prints the stored nickname and public key, confirming the upload succeeded.


In [ ]:
run_cli(["public-key", "info"])


## 6) Sign activity ID

Signs the full activity ID (`nickname.ActivityName+alias`) using the private key. The resulting signature is passed as a header when submitting workitems via `WorkItemAcc.run_public_activity(...)`.

> Replace `YourActivity` / `prod` to match your deployed public activity alias.


In [ ]:
activity_id = f"{APS_NICKNAME}.YourActivity+prod"
signature = run_cli([
    "signing",
    "sign",
    "--keyfile",
    str(PRIVATE_KEY),
    "--activity-id",
    activity_id,
])

print("Activity ID:", activity_id)
print("Signature:", signature)


## 7) Optional cleanup 

Deletes the key files generated in this notebook. The `keys/` directory is also removed if empty.

> To remove remote resources (Activity / AppBundle), use `delete_activity` / `delete_appbundle` from the SDK.


In [ ]:
for p in [PRIVATE_KEY, PUBLIC_KEY]:
    if p.exists():
        p.unlink()
        print("Deleted", p)

if KEY_DIR.exists() and not any(KEY_DIR.iterdir()):
    KEY_DIR.rmdir()
    print("Deleted empty", KEY_DIR)
